# 03 · Join Sofascore + Capology — Spain La Liga 20/21

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2020/21 de La Liga española**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2021.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2021.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  571 jugadores | 116 columnas
Capology:   545 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   deportivo alaves
   levante ud
   real valladolid

En Capology pero no en Sofascore:
   alaves
   levante
   valladolid


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'alaves':'deportivo alaves',
            'levante':'levante ud',
            'valladolid':'real valladolid'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 476/571 (83.4%)
Sin emparejar: 95


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          6
Revisión media    (0.75 ≤ score < 0.90):   10
Revisión estricta (0.50 ≤ score < 0.75):   52
Revisión muy est. (score < 0.50):           27


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
28,Martin Ødegaard,Real Madrid,martin odegaard,0.966
24,Jens Jønsson,Cádiz,jens jonsson,0.957
15,Javier Ontiveros,Huesca,javi ontiveros,0.933
39,Manuel Sánchez,Osasuna,manu sanchez,0.923
16,Yéremy Pino,Villarreal,yeremi pino,0.909
48,Dani Raba,Villarreal,daniel raba,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
34,Nico Williams,Athletic Club,inaki williams,0.815
76,Juan Carlos,Huesca,juan carlos real,0.815
7,José María Giménez,Atlético Madrid,jose gimenez,0.800
68,Waldo Rubio Marín,Real Valladolid,waldo rubio,0.786
17,José Luis Gayà,Valencia,jose gaya,0.783
36,Sergio Arribas,Real Madrid,sergio ramos,0.769
49,Fernando Niño,Villarreal,fer nino,0.762
82,Marc Baró,Cádiz,marcos mauro,0.762
25,Luis Javier Suárez,Granada,luis suarez,0.759
88,Koke,Levante UD,coke,0.750


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['nico williams',
                      'sergio arribas',
                      'marc baro'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 7 | Excluidos: 3


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
8,Maximiliano Gómez,Valencia,maxi gomez,0.741
1,Thomas Partey,Atlético Madrid,thomas lemar,0.720
27,Alejandro Baena,Villarreal,alex baena,0.720
52,Alberto,Cádiz,alberto perea,0.700
9,Emerson Royal,Real Betis,emerson,0.700
26,Papakouli Diop,Eibar,pape diop,0.696
41,Anthony Lozano,Cádiz,choco lozano,0.692
71,Ricard Sánchez,Atlético Madrid,manu sanchez,0.692
42,Sergio Carreira,Celta Vigo,sergio alvarez,0.690
90,Hugo Sotelo,Celta Vigo,hugo mallo,0.667


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['maximiliano gomez',
                    'alejandro baena',
                    'alberto',
                    'emerson royal',
                    'papakouli diop',
                    'anthony lozano',
                    'roberto jimenez',
                    'francisco trincao',
                    'quique'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 9


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
85,Aritz Arambarri,Real Sociedad,asier illarramendi,0.485
46,Miguel Atienza,Eibar,quique gonzalez,0.483
13,Vicente Esquerdo,Valencia,kevin gameiro,0.483
51,Urko González,Real Sociedad,carlos fernandez,0.483
61,Stephane Keller,Deportivo Alavés,jota peleteiro,0.483
44,Álejandro Cantero,Levante UD,carlos clerc,0.483
78,Koba Koindredi,Valencia,kang in lee,0.480
4,Sergio Barcia,Granada,adrian marin,0.480
31,Oriol Rey,Real Valladolid,marcos andre,0.476
87,Eñaut Mendia,Eibar,paulo oliveira,0.462


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 498/571 (87.2%)
Sin salario:     73


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 73


,player,team,minutesPlayed,appearances,goals,assists
0,Nico Williams,Athletic Club,55,2,0,0
1,Thomas Partey,Atlético Madrid,199,3,0,0
2,Ricard Sánchez,Atlético Madrid,29,1,0,0
3,Ilaix Moriba,Barcelona,538,14,1,2
4,Jose Fontán,Celta Vigo,660,14,0,0
5,Carlos Domínguez,Celta Vigo,360,4,0,0
6,Facundo Ferreyra,Celta Vigo,313,13,1,0
7,Sergio Carreira,Celta Vigo,261,3,1,0
8,Jordan Holsgrove,Celta Vigo,141,5,0,0
9,Alfon González,Celta Vigo,32,2,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Athletic Club  —  SF sin salario:


,player,minutesPlayed
0,Nico Williams,55


  CG plantilla completa:


,player,player_norm
0,Álex Berenguer,alex berenguer
1,Ander Capa,ander capa
2,Asier Villalibre,asier villalibre
3,Dani García,dani garcia
4,Iago Herrerín,iago herrerin
5,Ibai Gómez,ibai gomez
6,Iker Muniain,iker muniain
7,Iñaki Williams,inaki williams
8,Iñigo Córdoba,inigo cordoba
9,Iñigo Lekue,inigo lekue



  Atlético Madrid  —  SF sin salario:


...

,player,player_norm
0,Álex Blanco,alex blanco
1,Carlos Soler,carlos soler
2,Christian Oliva,christian oliva
3,Cristian Rivero,cristian rivero
4,Cristiano Piccini,cristiano piccini
5,Daniel Wass,daniel wass
6,Denis Cheryshev,denis cheryshev
7,Eliaquim Mangala,eliaquim mangala
8,Ferro,ferro
9,Gabriel Paulista,gabriel paulista



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Álex Millán,1


  CG plantilla completa:


,player,player_norm
0,Alberto Moreno,alberto moreno
1,Álex Baena,alex baena
2,Alfonso Pedraza,alfonso pedraza
3,Carlos Bacca,carlos bacca
4,Dani Parejo,dani parejo
5,Daniel Raba,daniel raba
6,Étienne Capoue,etienne capoue
7,Fer Niño,fer nino
8,Francis Coquelin,francis coquelin
9,Gerard Moreno,gerard moreno


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('rodri sanchez', 'real betis'): ('rodri', 'real betis'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: rodri sanchez (real betis) → rodri (real betis)

Tras matches manuales: 499/571 (87.4%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [21]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2021.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2021.csv
   Jugadores totales:  571
   Con salario:        499
   Sin salario (NaN):  72
   Columnas:           121
